In [1]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

In [2]:
import pandas as pd
import numpy as np
import optuna
import jax.numpy as jnp

In [3]:
import sys
from pathlib import Path
# Add project root to path for imports
ROOT = Path.cwd().parent  
sys.path.append(str(ROOT))

from src.gradient_flows.potentials import params as default_params
from src.mectrics.ist import analyze_bulk_plays_safe

In [4]:
STUDY_NAME = "nba-defensive-optimization-v2"
STORAGE_NAME = f"sqlite:///{STUDY_NAME}.db"
study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE_NAME)

# 1. Start with the full set of defaults
# This ensures no KeyErrors for things like 'basket_gravity_sigma'
solver_params = default_params.copy()

# 2. Get the best trial
# 115 is good
target_trial_number = 115
best_trial = None
for trial in study.trials:
    if trial.number == target_trial_number:
        best_trial = trial
        break
#best_trial = min(study.best_trials, key=lambda t: t.values[0])

# 3. Merge the optimized values and your manual overrides into one dictionary
# We use the update() pattern to ensure we have a single, complete set of params
solver_params.update(best_trial.params)
solver_params.update({
    'blending_radius': jnp.array(15.0),
    'blending_k': jnp.array(0.5),
    'attraction_weight': 7.570765933969438,
    'basket_gravity_weight': 1.5510598901937178,
    'basket_gravity_sigma': jnp.array(12.0),
    'field_weight': -16.627957913865643,
    'global_ball_weight': 0.10151136411006659,
    'offender_threat_scale': jnp.array(2.0),
    'offender_threat_dist': jnp.array(15.0),
    'k_softmin': 7.211277690769142,
    'occupancy_weight': 4.060469919776791,
    'cohesion_weight': 1.192885663129121,
    'formation_radius': 19.943871757515588,
    'sigma_long': 4.2939688756721175,
    'sigma_wide': 2.7709911763599617,
    'cushion_dist': 2.1430822408443366,
    'learning_rate': 0.1,
    'sprint_penalty_weight': 85.28395263390695,
    'jko_lambda': 0.5,
    'sinkhorn_epsilon': 0.01,
    'velocity_cap': 0.8,
    'soft_velocity_cap': 0.6,
    'court_dims': [[0.0, 94.0], [0.0, 50.0]],
    'max_gradient_norm': 1.0,
    'acceleration_penalty_weight': 2.0,
    'velocity_penalty_weight': 1.0,
})

# Rename for your simulation function call
best_params = solver_params 

print(f"--- Loaded Trial {best_trial.number} ---")

--- Loaded Trial 115 ---


In [5]:
PROCESSED_DIR = "../data/processed/traj_features"
analyze_bulk_plays_safe(PROCESSED_DIR, best_params)

Found 631 games. Building Play-Level Master Dataset...


Games Processed:   0%|          | 0/631 [00:00<?, ?it/s]

  -> traj_10.30.2015.WAS.at.MIL_21500027.parquet: Extracted 123 (Dropped 3)
  -> traj_11.25.2015.MEM.at.HOU_21500220.parquet: Extracted 118 (Dropped 7)
  -> traj_11.07.2015.MIN.at.CHI_21500085.parquet: Extracted 151 (Dropped 2)
  -> traj_11.27.2015.WAS.at.BOS_21500229.parquet: Extracted 163 (Dropped 6)
  -> traj_10.30.2015.POR.at.PHX_21500032.parquet: Extracted 134 (Dropped 9)
  -> traj_11.08.2015.PHX.at.OKC_21500097.parquet: Extracted 169 (Dropped 6)
  -> traj_12.01.2015.PHX.at.BKN_21500264.parquet: Extracted 129 (Dropped 3)
  -> traj_12.09.2015.NYK.at.UTA_21500327.parquet: Extracted 138 (Dropped 0)
  -> traj_12.03.2015.IND.at.POR_21500281.parquet: Extracted 147 (Dropped 3)
  -> traj_12.02.2015.MIL.at.SAS_21500275.parquet: Extracted 141 (Dropped 2)
  -> traj_10.27.2015.CLE.at.CHI_21500002.parquet: Extracted 158 (Dropped 2)
  -> traj_12.16.2015.POR.at.OKC_21500378.parquet: Extracted 106 (Dropped 2)
  -> traj_12.12.2015.WAS.at.DAL_21500350.parquet: Extracted 138 (Dropped 1)
  -> traj_01

,Game_File,Play_Index,GAME_ID,SHOT_EVENT_ID,PERIOD,GAME_CLOCK,Offensive_Team_ID,Defensive_Team_ID,Shooter_PID,Shot_Dist,...,Defender_1_PID,Offender_2_PID,Defender_2_PID,Offender_3_PID,Defender_3_PID,Offender_4_PID,Defender_4_PID,Offender_5_PID,Defender_5_PID,Shot_Made
0,traj_10.30.2015.WAS.at.MIL_21500027.parquet,0,21500027,4,1,680,1610612749,1610612764,202328,3.28,...,2743,203114,101162,203948,202322,203487,203078,203507,203490,1
1,traj_10.30.2015.WAS.at.MIL_21500027.parquet,1,21500027,6,1,662,1610612764,1610612749,101162,7.90,...,202328,2743,203114,202322,203948,203078,203487,203490,203507,0
2,traj_10.30.2015.WAS.at.MIL_21500027.parquet,2,21500027,14,1,644,1610612749,1610612764,203507,19.08,...,2743,202328,101162,203114,202322,203948,203078,203487,203490,1
3,traj_10.30.2015.WAS.at.MIL_21500027.parquet,3,21500027,15,1,638,1610612764,1610612749,202322,7.92,...,202328,2743,203114,101162,203948,203078,203487,203490,203507,1
4,traj_10.30.2015.WAS.at.MIL_21500027.parquet,4,21500027,16,1,622,1610612749,1610612764,202328,15.12,...,2743,203114,101162,203948,202322,203487,203078,203507,203490,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87741,traj_01.20.2016.BOS.at.TOR_21500633.parquet,134,21500633,448,4,136,1610612761,1610612738,203082,26.14,...,101161,200768,202738,202335,202340,201942,203482,202685,203109,1
87742,traj_01.20.2016.BOS.at.TOR_21500633.parquet,135,21500633,463,4,100,1610612738,1610612761,202738,10.68,...,200768,101161,202335,202340,201942,203482,203082,203109,202687,0
87743,traj_01.20.2016.BOS.at.TOR_21500633.parquet,136,21500633,472,4,65,1610612738,1610612761,203109,14.45,...,200768,101161,202335,202738,201942,202340,203082,203482,202685,1
87744,traj_01.20.2016.BOS.at.TOR_21500633.parquet,137,21500633,473,4,46,1610612761,1610612738,201942,21.55,...,101161,200768,202738,202335,202340,203082,203482,202685,203109,1
